# Retrieval Augmented Generation (RAG)
Retrieval Augmented Generation (RAG) is a technique that helps you work with large documents that are too big to fit into a single prompt. Instead of cramming everything into one massive prompt, RAG breaks documents into chunks and only includes the most relevant pieces when answering questions

RAg breaks the document up into many chunks and puts chunks relevant to the users qustion in the prompt.


 We can have size, structure and semantic based chunking 

In [3]:
# Calling clude

from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()

model = "claude-sonnet-4-5"


In [10]:
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0


    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunk_text = text[start_idx:end_idx]
        chunks.append(chunk_text)

    start_idx = (
        end_idx - chunk_overlap if end_idx < len(text) else len(text)

    )
    return chunks

#### Structure based chunking

Structure-based chunking divides text based on the the  documents natural structure-headers, paragraphs, and sections. Works well with .md files 

In [11]:
def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

#### Sentence base chunking 
A pratical middle ground of the computationally expensive semantic chunking is sentence chunking.

In [12]:
def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    
    chunks = []
    start_idx = 0
    
    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
        current_chunk = sentences[start_idx:end_idx]
        chunks.append(" ".join(current_chunk))
        
        start_idx += max_sentences_per_chunk - overlap_sentences
        
        if start_idx < 0:
            start_idx = 0
    
    return chunks

### Text Embeddings 
NExt step is finding which chunks are most useful to the question

#### Semantic Search    
Uses text embeddings to understand the meaning and context of both the users questions and each text chunk


In [30]:
from dotenv import load_dotenv
import voyageai

load_dotenv()
client = voyageai.Client()

def generate_embedding(text, model="voyage-3-large", input_type="query"):
    result = client.embed([text], model=model, input_type=input_type)
    return result.embeddings[0]

In [31]:
import re

with open("Claude101.md") as f:
    text = f.read()

chunks = chunk_by_section(text)

generate_embedding(chunks[0])

[-0.026134802028536797,
 0.006337198428809643,
 0.0066810776479542255,
 0.009333858266472816,
 -0.017193948850035667,
 -0.04185498505830765,
 -0.018078209832310677,
 -0.007614463102072477,
 -0.04165848344564438,
 -0.007270583882927895,
 -0.031243862584233284,
 -0.016407940536737442,
 -0.02711731381714344,
 0.019650226458907127,
 -0.03517390415072441,
 -0.034584399312734604,
 0.028099825605750084,
 0.10218118131160736,
 -0.026134802028536797,
 -0.05109059065580368,
 0.009923364967107773,
 -0.07624287903308868,
 -0.022499511018395424,
 0.02397327683866024,
 -0.05187659710645676,
 -0.024071527644991875,
 0.02711731381714344,
 0.07545687258243561,
 0.004740617237985134,
 0.017979959025979042,
 -0.02574179694056511,
 -0.006582826375961304,
 -0.06445274502038956,
 0.047160547226667404,
 0.015720181167125702,
 -0.023187266662716866,
 -0.048536062240600586,
 -0.0019036157755181193,
 -0.02731381542980671,
 -0.042444489896297455,
 -0.06288072466850281,
 -0.029278840869665146,
 0.0047651799395680

#### The Full RAG flow:

Now that we've covered the basics of RAG, text chunking, and embeddings, let's walk through the complete RAG pipeline step by step

Step 1: Chunk Your Source Text
First, we take our source document and break it into manageable chunks. For this example, we'll use two simple text sections:

Section 1: Medical Research - "This year saw significant strides in our understanding of XDR-47, a 'bug' we have not seen before."
Section 2: Software Engineering - "This division dedicated significant effort to studying various infection vectors in our distributed systems"

Step 2: Generate Embeddings
Next, we convert each text chunk into numerical embeddings using an embedding model. To make this easier to understand, let's imagine we have a perfect embedding model that always returns exactly two numbers, and we know what each number represents

Normalization
The embedding API typically performs a normalization step that scales each vector to have a magnitude of 1.0. You don't need to worry about the math here - it's handled automatically. This gives us normalized vectors like [0.944, 0.331] and [0.295, 0.955].

Step 3: Store in Vector Database
We store these embeddings in a vector database - a specialized database optimized for storing, comparing, and searching through long lists of numbers like our embeddings.

Step 4: Process User Query
When a user asks a question like "I'm curious about the company. In particular, what did the software engineering dept do this year?", we run their query through the same embedding model.

Step 5: Find Similar Embeddings
We send the user's query embedding to our vector database and ask it to find the most similar stored embeddings.

How Similarity Works: Cosine Similarity
The vector database uses cosine similarity to determine which embeddings are most similar. This measures the cosine of the angle between two vectors.

Step 6: Create the Final Prompt
Finally, we take the user's question and the most relevant text chunk we found, combine them into a prompt, and send it to Claude for a response.


In [42]:
import re, voyageai

voyage_client = voyageai.Client()


def chunk_by_section(document_text):
    chunks = re.split(r"\n(?=## )", document_text)
    return [c.strip() for c in chunks if c.strip()]


def generate_embeddings(texts, input_type="document"):
    if isinstance(texts, str):
        texts = [texts]
    result = voyage_client.embed(texts, model="voyage-3-large", input_type=input_type)
    return result.embeddings


# Steps 1-2
with open("Claude101.md") as f:
    text = f.read()

chunks = chunk_by_section(text)
embeddings = generate_embeddings(chunks, input_type="document")
print(f"{len(chunks)} chunks, {len(embeddings[0])} dimensions")

4 chunks, 1024 dimensions


In [43]:
import numpy as np
##Step 3 the store


store = [{"content": c, "embedding": e} for c, e in zip(chunks, embeddings)]
matrix = np.array(embeddings)



In [46]:
user_embedding = generate_embedding("What is in the claude 101 course?")

print(user_embedding)

[-0.031122295185923576, 0.037998151034116745, -0.004093848168849945, 0.03546493873000145, 0.03003663197159767, -0.0488547645509243, -0.014385013841092587, -0.052473634481430054, -0.05102609097957611, 0.006152081768959761, -0.027322478592395782, 0.015199260786175728, -0.01112802978605032, 0.03854098170995712, -0.03618871420621872, -0.03256984427571297, 0.049216654151678085, 0.09119556844234467, -0.03836003690958023, -0.011263737455010414, 0.033836446702480316, -0.04994042590260506, -0.005450925324112177, 0.0021939408034086227, -0.040169473737478256, -0.014113598503172398, -0.011489917524158955, 0.04270268231630325, -0.013480296358466148, 0.06296836584806442, 0.0027480805292725563, -0.01293746568262577, -0.04179796576499939, 0.03347456082701683, -0.011082793585956097, 0.005677104461938143, -0.052473634481430054, 0.009770953096449375, -0.023522663861513138, -0.07418686151504517, -0.039264753460884094, -0.05211174860596657, -0.009680481627583504, -0.04197890684008598, 0.04577872157096863, 

In [52]:
def search(question, k=2):
    q = np.array(generate_embeddings(question, input_type="query")[0])

    # cosine similarity
    sims = matrix @ q / (np.linalg.norm(matrix, axis=1) * np.linalg.norm(q))
    top = np.argsort(sims)[::-1][:k]

    return [(store[i]["content"], sims[i]) for i in top]


for content, score in search("What is claude"):
    print(f"{score:.3f}\n{content[:200]}\n")

0.579
## Lesson 1 - What is claude?
### Objectives
In this Lesson:
 - Explain what claude is and the principles that guide its design.
 - Describe Claudes core capabilities and how it differs from a simple 

0.572
# Claude 101



In [62]:
from rank_bm25 import BM25Okapi

import re
from rank_bm25 import BM25Okapi


class BM25Index:
    def __init__(self):
        self.documents = []
        self.bm25 = None

    def _tokenize(self, text):
        return re.findall(r"\w+", text.lower())

    def add_document(self, doc):
        self.documents.append(doc)
        self.bm25 = BM25Okapi([self._tokenize(d["content"]) for d in self.documents])

    def search(self, query, k=3):
        scores = self.bm25.get_scores(self._tokenize(query))
        ranked = sorted(zip(self.documents, scores), key=lambda x: x[1], reverse=True)
        return ranked[:k]



In [67]:
store = [{"content": c, "embedding": e} for c, e in zip(chunks, embeddings)]

In [69]:
bm25_store = BM25Index()
for chunk in chunks:
    bm25_store.add_document({"content": chunk})

results = bm25_store.search("What is claude?", 3)
for doc, score in results:
    print(f"{score:.2f}\n{doc['content'][:200]}\n----")

0.59
## Lesson 1 - What is claude?
### Objectives
In this Lesson:
 - Explain what claude is and the principles that guide its design.
 - Describe Claudes core capabilities and how it differs from a simple 
----
0.50
## Lesson 3 - Getting better results 
### Objectives 
- Recognize common challenges when starting out with AI and use troubleshooting techniques to overcome them.
- Define AI Fluency and know where to
----
0.48
## Lesson 2 - First Convo with Claude
### Writing Effective Prompts
All interactions with claude begin with a prompt, and these prompts, combiend with other context, impact claudes response. Speak to 
----


#### A multi- index RAG pipeline

In [72]:
class Retriever:
    def __init__(self, *indexes):
        if len(indexes) == 0:
            raise ValueError("At least one index must be provided")
        self._indexes = list(indexes)

    def add_document(self, document):
        for index in self._indexes:
            index.add_document(document)

    def search(self, query_text, k=1, k_rrf=60):
        all_results = [index.search(query_text, k * 3) for index in self._indexes]

        scores = {}
        docs = {}

        for results in all_results:
            for rank, (doc, _) in enumerate(results):
                key = doc["content"]
                scores[key] = scores.get(key, 0) + 1 / (k_rrf + rank + 1)
                docs[key] = doc

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [(docs[key], score) for key, score in ranked[:k]]

In [74]:
import numpy as np


class VectorIndex:
    def __init__(self):
        self.documents = []
        self.vectors = []

    def add_document(self, doc):
        embedding = generate_embeddings(doc["content"], input_type="document")[0]
        self.documents.append(doc)
        self.vectors.append(embedding)

    def search(self, query, k=3):
        q = np.array(generate_embeddings(query, input_type="query")[0])
        matrix = np.array(self.vectors)

        sims = matrix @ q / (np.linalg.norm(matrix, axis=1) * np.linalg.norm(q))
        top = np.argsort(sims)[::-1][:k]

        return [(self.documents[i], float(sims[i])) for i in top]

In [75]:
retriever = Retriever(BM25Index(), VectorIndex())

for chunk in chunks:
    retriever.add_document({"content": chunk})

for doc, score in retriever.search("how do I stop the model rambling", 3):
    print(f"{score:.4f}  {doc['content'][:80]}")

RateLimitError: You have not yet added your payment method in the billing page and will have reduced rate limits of 3 RPM and 10K TPM. To unlock our standard rate limits, please add a payment method in the billing page for the appropriate organization in the user dashboard (https://dashboard.voyageai.com/). Even with payment methods entered, the free tokens (200M tokens for Voyage series 3) will still apply. After adding a payment method, you should see your rate limits increase after several minutes. See our pricing docs (https://docs.voyageai.com/docs/pricing) for the free tokens for your model.

### Had to stop because of self-budget costing too much 

### Extended thinking:

In [79]:
from anthropic.types import Message


def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })


def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })

In [80]:
def text_from_message(message):
    return "\n".join(b.text for b in message.content if b.type == "text")

In [85]:
def chat(messages, system=None, stop_sequences=None, tools=None,
         thinking=False, thinking_budget=1024, max_tokens=4000):
    params = {"model": model, "max_tokens": max_tokens, "messages": messages}

    if thinking:
        params["max_tokens"] = thinking_budget + 1000
        params["thinking"] = {"type": "enabled", "budget_tokens": thinking_budget}

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if tools:
        params["tools"] = tools

    return anthropic_client.messages.create(**params)

In [83]:
from anthropic import Anthropic

anthropic_client = Anthropic()

In [86]:
messages = []
add_user_message(messages, "A bat and ball cost £1.10 together. The bat costs £1 more than the ball. How much is the ball?")

response = chat(messages, thinking=True)
print(text_from_message(response))

Looking at this step-by-step:

Let me call the ball's price **x**.

Since the bat costs £1 more than the ball, the bat costs **x + £1**.

Together they cost £1.10, so:
- x + (x + £1) = £1.10
- 2x + £1 = £1.10
- 2x = £0.10
- x = £0.05

**The ball costs £0.05 (5 pence)**

To check: Ball = £0.05, Bat = £1.05, Total = £1.10 ✓ and the bat costs £1 more ✓


### Image Support

In [89]:
import base64
from pathlib import Path


def image_block(path):
    data = base64.standard_b64encode(Path(path).read_bytes()).decode("utf-8")
    suffix = Path(path).suffix.lower()
    media_type = {"jpg": "image/jpeg", "jpeg": "image/jpeg",
                  "png": "image/png", "gif": "image/gif",
                  "webp": "image/webp"}[suffix.lstrip(".")]

    return {
        "type": "image",
        "source": {"type": "base64", "media_type": media_type, "data": data},
    }

In [91]:
prompt = """Analyse this image using these steps:
1. Describe the overall composition.
2. List every distinct element you can identify.
3. Note any text and transcribe it exactly.

Summarise your findings in one paragraph."""

In [92]:
messages = []
add_user_message(messages, [
    image_block("Claue101_badge.png"),
    {"type": "text", "text": prompt},
])

response = chat(messages)
print(text_from_message(response))

# Image Analysis

## 1. Overall Composition
This is a digital certificate or badge with a salmon/coral-colored background. The central element is a large beige/tan dodecagon (12-sided polygon) containing the certificate information. The design is clean and modern with a professional aesthetic.

## 2. Distinct Elements
- Salmon/coral-colored background
- Large beige dodecagonal central shape
- Small icon of two overlapping speech bubbles or chat heads
- Curved text along the top arc
- Large bold serif title text
- Subtitle text
- Recipient name
- Footer with logo/branding
- Date stamp in bottom right

## 3. Text Transcription
- "COURSE COMPLETION BADGE" (curved text at top)
- "Claude 101" (large title)
- "Presented to"
- "Harry"
- "Claude Academy" (bottom left with sunburst icon)
- "ISSUED SEPTEMBER 14, 2025" (bottom right)

## Summary
This is a course completion certificate badge awarded by Claude Academy to someone named Harry for completing "Claude 101." The design features a dodecag